# Lean Statement Diversity Dataset — Perturbation Pipeline
Rule-based transforms — no API key required. Produces `(anchor, variant, transformation_type)` triples.

In [ ]:
import re, json, math, subprocess, tempfile, os
import pandas as pd
from pathlib import Path
from huggingface_hub import HfFileSystem
from itertools import permutations
from enum import Enum
from typing import Optional
import random
from bounds import flip_bound, perturb_bound
from typeclass_mutate import weaken_statement_by_weakening_conclusion, weaken_statement_by_strengthening_hypotheses, generalize_statement_by_weakening_hypotheses, strengthen_statement_by_strengthening_conclusion

fs = HfFileSystem()
indices = [5145, 181597, 196429, 14831, 101254, 172154, 44743, 60309, 44790, 63001]

OUTPUT_FILE = Path("perturbation_pairs.jsonl")

In [13]:
df = pd.read_json(
    "hf://datasets/FrenzyMath/mathlib_informal_v4.19.0/data.jsonl",
    lines=True,
)

In [14]:
# df = df.iloc[indices]
df = df[0:10]

df = df[['signature', 'type']]
df.head()

,signature,type
0,(A : Hopf_ C) :\n A.X.comul.hom ≫\n A.X...,∀ {C : Type u₁} [inst : CategoryTheory.Categor...
1,{W : C} (k : Y ⟶ W) (h : f ≫ k = g ≫ k) : ∃! ...,∀ {C : Type u} [inst : CategoryTheory.Category...
2,{P : C} (ι : P ⟶ X) (w : ι ≫ f = ι ≫ g) : (Fo...,∀ {C : Type u} [inst : CategoryTheory.Category...
3,{P : C} {ι ι' : P ⟶ X} (w : ι ≫ f = ι ≫ g) (w...,{C : Type u} →\n [inst : CategoryTheory.Categ...
4,(f g : X ⟶ Y) : (parallelPair f g).obj one = Y,∀ {C : Type u} [inst : CategoryTheory.Category...


In [15]:
# ── Lean runner ───────────────────────────────────────────────────────────────
PROJECT_DIR = os.path.abspath(".")
_TMP_LEAN = os.path.join(PROJECT_DIR, "_tmp_nb.lean")

def _run_lean(code: str) -> str:
    try:
        with open(_TMP_LEAN, "w") as f:
            f.write(code)
        result = subprocess.run(
            ["lake", "env", "lean", _TMP_LEAN],
            cwd=PROJECT_DIR,
            capture_output=True,
            text=True,
            timeout=300,
        )
        return result.stdout + result.stderr
    finally:
        if os.path.exists(_TMP_LEAN):
            os.remove(_TMP_LEAN)

def compile_lean(variant_sig: str, variant_type: str) -> bool:
    output = _run_lean(f"import Mathlib\n\nexample : {variant_type} := by sorry\n")
    return "error:" not in output


In [ ]:
# ── is_true ───────────────────────────────────────────────────────────────────
_PROPAGATION_RULES = {
    "negation":                  {"true": "false",   "false": "true",    "unknown": "unknown"},
    "contrapositive":            {"true": "true",    "false": "false",   "unknown": "unknown"},
    "converse":                  {"true": "unknown", "false": "unknown", "unknown": "unknown"},
    "specialization":            {"true": "true",    "false": "unknown", "unknown": "unknown"},
    "generalization":            {"true": "unknown", "false": "false",   "unknown": "unknown"},
    "bound_perturbation_looser": {"true": "true",    "false": "unknown", "unknown": "unknown"},
    "bound_perturbation_tighter":{"true": "unknown", "false": "false",   "unknown": "unknown"},
    "flip_bound":                {"true": "unknown", "false": "unknown", "unknown": "unknown"},
    "perturb_bound":             {"true": "unknown", "false": "false",   "unknown": "unknown"},
    # Weaken hypotheses → strictly fewer assumptions → original proof still holds
    "generalize_statement_by_weakening_hypotheses":    {"true": "unknown", "false": "false",   "unknown": "unknown"},
    # Strengthen hypotheses → strictly more assumptions → original conclusion still follows
    "weaken_statement_by_strengthening_hypotheses":    {"true": "true",    "false": "unknown", "unknown": "unknown"},
    # Strengthen conclusion → harder to prove → original truth no longer guaranteed
    "strengthen_statement_by_strengthening_conclusion":{"true": "unknown", "false": "false",   "unknown": "unknown"},
    # Weaken conclusion → easier to prove → if original was true, this is too
    "weaken_statement_by_weakening_conclusion":        {"true": "true",    "false": "unknown", "unknown": "unknown"},
}

def is_true(perturbation_path: list[str], variant_sig: str, variant_type: str) -> str:
    """
    Propagate truth symbolically through the perturbation path.
    Anchors are always 'true' (they come from Mathlib — all proven theorems).
    Returns one of: "true", "false", "unknown"
    """
    current = "true"
    for perturbation in perturbation_path:
        rules = _PROPAGATION_RULES.get(perturbation)
        current = rules[current] if rules else "unknown"
    return current

In [ ]:
# ── Perturbation functions ────────────────────────────────────────────────────
def _extract_goal(tactic: str, type_str: str) -> tuple[str, str] | None:
    """
    Shared helper: runs a Lean snippet with the given tactic, scrapes the
    extract_goal output, and returns (variant_sig, variant_type).
    """
    output = _run_lean(
        f"import Mathlib\nimport Wiggle\n\nexample : {type_str} := by\n  {tactic}\n  extract_goal\n  sorry\n"
    )
    m = re.search(r"^theorem .*extracted.*$", output, re.MULTILINE)
    if m is None:
        return None
    full_statement = m.group(0)
    without_proof = full_statement.rsplit(":= sorry", 1)[0].strip()
    parts = without_proof.split(" : ", 1)
    if len(parts) != 2:
        return None
    variant_sig, variant_type = parts
    return variant_sig.strip(), variant_type.strip()


def negate(sig: str, type_str: str) -> tuple[str, str] | None:
    return _extract_goal("negate_state", type_str)


def contrapositive(sig: str, type_str: str) -> tuple[str, str] | None:
    return _extract_goal("contrapositive", type_str)


def converse(sig: str, type_str: str) -> tuple[str, str] | None:
    return _extract_goal("converse", type_str)


def generalize(sig: str, type_str: str) -> tuple[str, str] | None:
    return _extract_goal("generalize_state", type_str)


def specialize(sig: str, type_str: str) -> tuple[str, str] | None:
    return _extract_goal("specialize_state", type_str)

def generalize_statement_by_weakening_hypotheses_wrapper(
    sig: str, type_str: str
) -> tuple[str, str] | None:
    result = generalize_statement_by_weakening_hypotheses(sig, type_str)
    if result is None:
        return None
    variant_sig, variant_type = result
    if variant_sig.strip() == sig.strip() and variant_type.strip() == type_str.strip():
        return None
    return variant_sig, variant_type


def weaken_statement_by_strengthening_hypotheses_wrapper(
    sig: str, type_str: str
) -> tuple[str, str] | None:
    result = weaken_statement_by_strengthening_hypotheses(sig, type_str)
    if result is None:
        return None
    variant_sig, variant_type = result
    if variant_sig.strip() == sig.strip() and variant_type.strip() == type_str.strip():
        return None
    return variant_sig, variant_type


def strengthen_statement_by_strengthening_conclusion_wrapper(
    sig: str, type_str: str
) -> tuple[str, str] | None:
    result = strengthen_statement_by_strengthening_conclusion(sig, type_str)
    if result is None:
        return None
    variant_sig, variant_type = result
    if variant_sig.strip() == sig.strip() and variant_type.strip() == type_str.strip():
        return None
    return variant_sig, variant_type


def weaken_statement_by_weakening_conclusion_wrapper(
    sig: str, type_str: str
) -> tuple[str, str] | None:
    result = weaken_statement_by_weakening_conclusion(sig, type_str)
    if result is None:
        return None
    variant_sig, variant_type = result
    if variant_sig.strip() == sig.strip() and variant_type.strip() == type_str.strip():
        return None
    return variant_sig, variant_type

TRANSFORMS = {
    "negate":        negate,
    "converse":      converse,
    "generalize":    generalize,
    "perturb_bound": perturb_bound,
    "flip_bound":    flip_bound,
    "generalize_statement_by_weakening_hypotheses":    generalize_statement_by_weakening_hypotheses_wrapper,
    "weaken_statement_by_strengthening_hypotheses":    weaken_statement_by_strengthening_hypotheses_wrapper,
    "strengthen_statement_by_strengthening_conclusion": strengthen_statement_by_strengthening_conclusion_wrapper,
    "weaken_statement_by_weakening_conclusion":         weaken_statement_by_weakening_conclusion_wrapper,
}

In [18]:
# ── apply_perturbation_chains ─────────────────────────────────────────────────
def apply_perturbation_chains(
    df: pd.DataFrame,
    transforms: dict,
    n_permutations: int = 6,
    random_seed: int = 42,
) -> pd.DataFrame:
    rng = random.Random(random_seed)
    transform_names = list(transforms.keys())
    all_permutations = list(permutations(transform_names))

    results = []

    for _, row in df.iterrows():
        anchor_sig  = row["signature"]
        anchor_type = row["type"]

        k = min(n_permutations, len(all_permutations))
        sampled_permutations = rng.sample(all_permutations, k)

        for perm in sampled_permutations:
            current_sig  = anchor_sig
            current_type = anchor_type
            applied_so_far = []

            for transform_name in perm:
                fn = transforms[transform_name]

                result = fn(current_sig, current_type)
                if result is None:
                    break

                variant_sig, variant_type = result
                if (variant_sig.strip() == current_sig.strip()
                        and variant_type.strip() == current_type.strip()):
                    break

                if not compile_lean(variant_sig, variant_type):
                    break

                # Compiled — save this intermediate as a datapoint
                applied_so_far.append(transform_name)
                results.append({
                    **{k: v for k, v in row.items() if k not in ("signature", "type")},
                    "anchor_signature":      anchor_sig,
                    "anchor_type":           anchor_type,
                    "variant_signature":     variant_sig,
                    "variant_type":          variant_type,
                    "perturbations_applied": list(applied_so_far),
                    "chain_depth":           len(applied_so_far),
                    "is_true":               is_true(list(applied_so_far), variant_sig, variant_type),
                })

                current_sig  = variant_sig
                current_type = variant_type

    result_df = pd.DataFrame(results)
    print(f"Generated {len(result_df):,} compiled variants from {len(df):,} anchors")
    if len(result_df):
        print(result_df["chain_depth"].value_counts().sort_index().to_string())
    return result_df
